# ☁ Cloud Resource Manager

### Mini Cloud Manager menggunakan AWS EC2 dan Amazon S3

---

Notebook ini digunakan untuk mensimulasikan pengelolaan infrastruktur cloud menggunakan:

- Amazon EC2
- Amazon S3
- Boto3
- Ministack / LocalStack

### Alur Sistem

1. Konfigurasi AWS
2. Infrastruktur Jaringan
3. Dashboard Monitoring
4. Manajemen EC2
5. Manajemen S3 Bucket
6. Manajemen Object Storage

In [1]:
# Import Library
import boto3
import pandas as pd

import ipywidgets as widgets

from IPython.display import display
from IPython.display import clear_output
from IPython.display import HTML

In [2]:
header = widgets.HTML("""
<div style="
    background: linear-gradient(90deg,#1565C0,#1976D2,#42A5F5);
    color:white;
    padding:25px;
    border-radius:12px;
    box-shadow:0 4px 10px rgba(0,0,0,0.25);
">

<h1 style="margin:0;">
☁ Cloud Resource Manager
</h1>

<h3 style="margin-top:8px;">
       Simulation Platform for Amazon EC2 and Amazon S3 using MiniStack
</h3>

<hr style="border:1px solid rgba(255,255,255,.4);">

<table style="width:100%; color:white;">
<tr>
<td width="40%">
<b>Nama</b><br>
M. Kharis Fadlussalam
</td>

<td width="30%">
<b>Mata Kuliah</b><br>
Cloud Computing
</td>

<td>
<b>Tahun Akademik</b><br>
2026/2027
</td>
</tr>
</table>

</div>
""")

display(header)

HTML(value='\n<div style="\n    background: linear-gradient(90deg,#1565C0,#1976D2,#42A5F5);\n    color:white;\…

# 1. Konfigurasi AWS

Menghubungkan notebook dengan Ministack sehingga seluruh layanan EC2 dan S3 dapat digunakan.

In [3]:
# ===============================
# Konfigurasi AWS
# ===============================

endpoint = "http://localhost:4566"
region = "us-east-1"

credentials = {
    "aws_access_key_id": "121212121212",
    "aws_secret_access_key": "admin123"
}

# Membuat Client EC2 dan S3
ec2 = boto3.client(
    "ec2",
    endpoint_url=endpoint,
    region_name=region,
    **credentials
)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    region_name=region,
    **credentials
)

print("✅ Berhasil Terhubung Ke Ministack")

✅ Berhasil Terhubung Ke Ministack


# 2. Infrastruktur Cloud

Tahap ini digunakan untuk membangun infrastruktur dasar yang akan digunakan oleh seluruh resource cloud.

Komponen yang dibuat:

- VPC
- Public Subnet
- Private Subnet

In [4]:
# ==========================================
# RESOURCE INFRASTRUCTURE
# ==========================================

# Menyimpan ID resource yang telah dibuat
vpc_id = None
public_subnet_id = None
private_subnet_id = None

In [5]:
# ==========================================
# INFRASTRUCTURE FUNCTION
# ==========================================

def create_vpc():
    """Membuat Virtual Private Cloud (VPC)"""

    global vpc_id

    try:
        response = ec2.create_vpc(
            CidrBlock="10.0.0.0/16"
        )

        vpc_id = response["Vpc"]["VpcId"]

        print("✅ VPC berhasil dibuat")
        print(f"VPC ID : {vpc_id}")

    except Exception as e:
        print(f"❌ Gagal membuat VPC\n{e}")


def create_public_subnet():
    """Membuat Public Subnet"""

    global public_subnet_id

    if vpc_id is None:
        print("❌ Buat VPC terlebih dahulu.")
        return

    try:
        response = ec2.create_subnet(
            VpcId=vpc_id,
            CidrBlock="10.0.1.0/24",
            AvailabilityZone="us-east-1"
        )

        public_subnet_id = response["Subnet"]["SubnetId"]

        print("✅ Public Subnet berhasil dibuat")
        print(f"Subnet ID : {public_subnet_id}")

    except Exception as e:
        print(e)


def create_private_subnet():
    """Membuat Private Subnet"""

    global private_subnet_id

    if vpc_id is None:
        print("❌ Buat VPC terlebih dahulu.")
        return

    try:
        response = ec2.create_subnet(
            VpcId=vpc_id,
            CidrBlock="10.0.2.0/24",
            AvailabilityZone="us-east-1"
        )

        private_subnet_id = response["Subnet"]["SubnetId"]

        print("✅ Private Subnet berhasil dibuat")
        print(f"Subnet ID : {private_subnet_id}")

    except Exception as e:
        print(e)


def show_infrastructure():
    """Menampilkan informasi infrastruktur"""

    print("=" * 45)
    print("     CLOUD INFRASTRUCTURE")
    print("=" * 45)

    print(f"VPC             : {vpc_id}")
    print(f"Public Subnet   : {public_subnet_id}")
    print(f"Private Subnet  : {private_subnet_id}")

In [6]:
# ==========================================
# INFRASTRUCTURE WIDGET
# ==========================================

infra_output = widgets.Output()

btn_vpc = widgets.Button(
    description="Create VPC",
    button_style="success",
    icon="server"
)

btn_public_subnet = widgets.Button(
    description="Public Subnet",
    button_style="info",
    icon="network-wired"
)

btn_private_subnet = widgets.Button(
    description="Private Subnet",
    button_style="warning",
    icon="lock"
)

btn_show_infra = widgets.Button(
    description="Show Infrastructure",
    button_style="primary",
    icon="list"
)

In [7]:
# ==========================================
# EVENT HANDLER
# ==========================================

def klik_vpc(b):
    with infra_output:
        infra_output.clear_output()
        create_vpc()


def klik_public_subnet(b):
    with infra_output:
        infra_output.clear_output()
        create_public_subnet()


def klik_private_subnet(b):
    with infra_output:
        infra_output.clear_output()
        create_private_subnet()


def klik_show_infrastructure(b):
    with infra_output:
        infra_output.clear_output()
        show_infrastructure()


# Bind Event
btn_vpc.on_click(klik_vpc)
btn_public_subnet.on_click(klik_public_subnet)
btn_private_subnet.on_click(klik_private_subnet)
btn_show_infra.on_click(klik_show_infrastructure)


# ==========================================
# LAYOUT
# ==========================================

infra_panel = widgets.VBox([

    widgets.HTML("<h2>🌐 Cloud Infrastructure</h2>"),

    widgets.HBox([
        btn_vpc,
        btn_public_subnet
    ]),

    widgets.HBox([
        btn_private_subnet,
        btn_show_infra
    ]),

    infra_output

])

display(infra_panel)

# 3. Dashboard Monitoring

Dashboard digunakan untuk menampilkan ringkasan kondisi seluruh resource cloud yang sedang berjalan.

### Informasi yang ditampilkan

- Total EC2 Instance
- Running Instance
- Stopped Instance
- Total Bucket S3
- Endpoint AWS

In [8]:
# OUPUT AREA

output = widgets.Output()

display(output)

Output()

In [9]:
# ==========================================
# DASHBOARD FUNCTION
# ==========================================

def dashboard():
    """Menampilkan ringkasan resource cloud"""

    with dashboard_output:

        dashboard_output.clear_output()

        # ==========================
        # EC2
        # ==========================
        instances = ec2.describe_instances()

        total_instance = 0
        running = 0
        stopped = 0

        for reservation in instances["Reservations"]:

            for instance in reservation["Instances"]:

                total_instance += 1

                if instance["State"]["Name"] == "running":
                    running += 1
                else:
                    stopped += 1

        # ==========================
        # S3
        # ==========================
        buckets = s3.list_buckets()
        total_bucket = len(buckets["Buckets"])

        # ==========================
        # OUTPUT
        # ==========================
        print("=" * 45)
        print("        CLOUD DASHBOARD")
        print("=" * 45)

        print(f"🖥 Total Instance : {total_instance}")
        print(f"🟢 Running        : {running}")
        print(f"🔴 Stopped        : {stopped}")

        print()

        print(f"🪣 Total Bucket   : {total_bucket}")

        print()

        print(f"☁ Endpoint       : {endpoint}")

In [10]:
# ==========================================
# DASHBOARD WIDGET
# ==========================================

dashboard_output = widgets.Output()

btn_dashboard = widgets.Button(
    description="Dashboard",
    icon="chart-bar",
    button_style="primary",
    layout=widgets.Layout(width="170px")
)

btn_ec2 = widgets.Button(
    description="EC2 Manager",
    icon="server",
    button_style="success",
    layout=widgets.Layout(width="170px")
)

btn_bucket = widgets.Button(
    description="S3 Manager",
    icon="database",
    button_style="warning",
    layout=widgets.Layout(width="170px")
)

btn_refresh_dashboard = widgets.Button(
    description="Refresh",
    icon="refresh",
    button_style="info",
    layout=widgets.Layout(width="170px")
)

In [11]:
# ==========================================
# DASHBOARD EVENT
# ==========================================

def klik_dashboard(b):
    dashboard()


def klik_refresh_dashboard(b):
    dashboard()


btn_dashboard.on_click(klik_dashboard)
btn_refresh_dashboard.on_click(klik_refresh_dashboard)

In [12]:
# ==========================================
# DASHBOARD LAYOUT
# ==========================================

dashboard_menu = widgets.HBox([
    btn_dashboard,
    btn_ec2,
    btn_bucket,
    btn_refresh_dashboard
])

dashboard_panel = widgets.VBox([

    widgets.HTML("<h2>📊 Dashboard Monitoring</h2>"),

    dashboard_menu,

    widgets.HTML("<hr>"),

    dashboard_output

])

display(dashboard_panel)

# Menampilkan dashboard saat pertama kali dibuka
dashboard()

# 4. EC2 Manager

Modul ini digunakan untuk mengelola Virtual Machine (Amazon EC2).

### Fitur

- Create Instance
- Menampilkan Daftar Instance
- Start Instance
- Stop Instance
- Delete Instance

In [13]:
# ==========================================
# EC2 WIDGET
# ==========================================

ec2_output = widgets.Output()
instance_table = widgets.Output()

instance_dropdown = widgets.Dropdown(
    options=[],
    description="Instance",
    layout=widgets.Layout(width="500px")
)

instance_type = widgets.Dropdown(
    options=[
        ("t2.micro", "t2.micro"),
        ("t2.small", "t2.small"),
        ("t2.medium", "t2.medium"),
        ("t3.micro", "t3.micro"),
        ("c8g.medium", "c8g.medium")
    ],
    value="c8g.medium",
    description="Type",
    layout=widgets.Layout(width="350px")
)

btn_create_instance = widgets.Button(
    description="➕ Create",
    button_style="success"
)

btn_refresh_instance = widgets.Button(
    description="🔄 Refresh",
    button_style="info"
)

btn_start_instance = widgets.Button(
    description="▶ Start",
    button_style="success"
)

btn_stop_instance = widgets.Button(
    description="⏸ Stop",
    button_style="warning"
)

btn_delete_instance = widgets.Button(
    description="🗑 Delete",
    button_style="danger"
)

## Fungsi EC2

In [14]:
# ==========================================
# EC2 FUNCTION
# ==========================================

def create_instance():

    if public_subnet_id is None:
        print("❌ Public Subnet belum dibuat.")
        return

    try:

        response = ec2.run_instances(
            ImageId="ami-mock-ubuntu",
            InstanceType=instance_type.value,
            MinCount=1,
            MaxCount=1,
            SubnetId=public_subnet_id
        )

        instance = response["Instances"][0]

        print("✅ Instance berhasil dibuat")
        print(f"Instance ID : {instance['InstanceId']}")
        print(f"Status      : {instance['State']['Name']}")

    except Exception as e:
        print(e)


def start_instance(instance_id):

    try:

        ec2.start_instances(
            InstanceIds=[instance_id]
        )

        print("✅ Instance dijalankan.")

    except Exception as e:
        print(e)


def stop_instance(instance_id):

    try:

        ec2.stop_instances(
            InstanceIds=[instance_id]
        )

        print("✅ Instance dihentikan.")

    except Exception as e:
        print(e)


def delete_instance(instance_id):

    try:

        ec2.terminate_instances(
            InstanceIds=[instance_id]
        )

        print("✅ Instance dihapus.")

    except Exception as e:
        print(e)


def refresh_instance_data():

    rows = []
    options = []

    reservations = ec2.describe_instances()["Reservations"]

    for reservation in reservations:

        for instance in reservation["Instances"]:

            status = instance["State"]["Name"]

            rows.append({

                "Instance ID": instance["InstanceId"],
                "Status": "🟢 Running" if status == "running" else "🔴 Stopped",
                "Type": instance["InstanceType"],
                "Subnet": instance.get("SubnetId", "-")

            })

            options.append((
                f"{instance['InstanceId']} ({status})",
                instance["InstanceId"]
            ))

    with instance_table:

        instance_table.clear_output()

        if rows:

            df = pd.DataFrame(rows)

            try:
                display(df.style.hide(axis="index"))
            except:
                display(df)

        else:
            print("Belum ada Instance.")

    instance_dropdown.options = (
        options if options
        else [("Belum ada Instance", None)]
    )

## Event Handler

In [15]:
# ==========================================
# EC2 EVENT
# ==========================================

def klik_create_instance(b):

    with ec2_output:

        ec2_output.clear_output()

        create_instance()

        refresh_instance_data()


def klik_refresh_instance(b):

    refresh_instance_data()


def klik_start_instance(b):

    with ec2_output:

        ec2_output.clear_output()

        if instance_dropdown.value:

            start_instance(instance_dropdown.value)

            refresh_instance_data()


def klik_stop_instance(b):

    with ec2_output:

        ec2_output.clear_output()

        if instance_dropdown.value:

            stop_instance(instance_dropdown.value)

            refresh_instance_data()


def klik_delete_instance(b):

    with ec2_output:

        ec2_output.clear_output()

        if instance_dropdown.value:

            delete_instance(instance_dropdown.value)

            refresh_instance_data()


btn_create_instance.on_click(klik_create_instance)
btn_refresh_instance.on_click(klik_refresh_instance)
btn_start_instance.on_click(klik_start_instance)
btn_stop_instance.on_click(klik_stop_instance)
btn_delete_instance.on_click(klik_delete_instance)

## Tampilan EC2 Manager

In [16]:
# ==========================================
# EC2 LAYOUT
# ==========================================

ec2_panel = widgets.VBox([

    widgets.HTML("<h2>🖥 EC2 Manager</h2>"),

    instance_type,

    widgets.HBox([
        btn_create_instance,
        btn_refresh_instance
    ]),

    widgets.HTML("<hr>"),

    instance_dropdown,

    instance_table,

    widgets.HBox([
        btn_start_instance,
        btn_stop_instance,
        btn_delete_instance
    ]),

    ec2_output

])

display(ec2_panel)

refresh_instance_data()

# 5. S3 Bucket Manager

Tahap ini digunakan untuk mengelola Bucket sebagai media penyimpanan object.

### Fitur

- Membuat Bucket
- Menghapus Bucket
- Upload Object
- Download Object
- Delete Object
- Refresh Data

In [17]:
# ==========================================
# WIDGET S3
# ==========================================

# Output Widget
bucket_output = widgets.Output()
bucket_table = widgets.Output()
object_table = widgets.Output()
object_output = widgets.Output()

# Input Bucket
bucket_name = widgets.Text(
    description="Bucket",
    placeholder="contoh: data-akademik",
    layout=widgets.Layout(width="450px")
)

# Dropdown Bucket
bucket_dropdown = widgets.Dropdown(
    options=[],
    description="Bucket",
    layout=widgets.Layout(width="450px")
)

# Dropdown Object
object_dropdown = widgets.Dropdown(
    options=[],
    description="Object",
    layout=widgets.Layout(width="450px")
)

# Upload Widget
upload_widget = widgets.FileUpload(
    accept="",
    multiple=False
)

# Tombol Bucket
btn_bucket_create = widgets.Button(
    description="➕ Create",
    button_style="success"
)

btn_bucket_refresh = widgets.Button(
    description="🔄 Refresh",
    button_style="info"
)

btn_bucket_delete = widgets.Button(
    description="🗑 Delete Bucket",
    button_style="danger"
)

# Tombol Object
btn_upload = widgets.Button(
    description="📤 Upload",
    button_style="success"
)

btn_download = widgets.Button(
    description="📥 Download",
    button_style="info"
)

btn_delete_object = widgets.Button(
    description="🗑 Delete Object",
    button_style="danger"
)

In [18]:
# ==========================================
# FUNGSI BUCKET
# ==========================================

def create_bucket():
    """Membuat bucket baru"""

    if not bucket_name.value.strip():
        print("Masukkan nama bucket.")
        return

    try:
        s3.create_bucket(
            Bucket=bucket_name.value.strip()
        )

        print(f"✅ Bucket '{bucket_name.value}' berhasil dibuat.")

        refresh_bucket_data()

    except Exception as e:
        print(e)


def refresh_bucket_data():
    """Menampilkan daftar bucket"""

    rows = []
    options = []

    try:
        response = s3.list_buckets()

        for bucket in response["Buckets"]:

            rows.append({
                "Bucket": bucket["Name"]
            })

            options.append(
                (bucket["Name"], bucket["Name"])
            )

    except Exception as e:
        print(e)

    with bucket_table:

        bucket_table.clear_output()

        if rows:

            df = pd.DataFrame(rows)

            try:
                display(df.style.hide(axis="index"))
            except:
                display(df)

        else:
            print("Belum ada bucket.")

    bucket_dropdown.options = (
        options if options
        else [("Belum ada Bucket", None)]
    )

    if bucket_dropdown.value:
        refresh_object_data()


def delete_bucket():
    """Menghapus bucket"""

    if bucket_dropdown.value is None:
        print("Pilih bucket.")
        return

    try:

        s3.delete_bucket(
            Bucket=bucket_dropdown.value
        )

        print("✅ Bucket berhasil dihapus.")

        refresh_bucket_data()

    except Exception as e:
        print(e)

In [19]:
# ==========================================
# OBJECT STORAGE FUNCTION
# ==========================================

def refresh_object_data():
    """Menampilkan daftar object pada bucket"""

    if bucket_dropdown.value is None:
        return

    rows = []
    options = []

    try:
        response = s3.list_objects_v2(
            Bucket=bucket_dropdown.value
        )

        if "Contents" in response:

            for obj in response["Contents"]:

                rows.append({
                    "Nama File": obj["Key"],
                    "Ukuran (Byte)": obj["Size"]
                })

                options.append((
                    obj["Key"],
                    obj["Key"]
                ))

    except Exception:
        pass

    with object_table:

        object_table.clear_output()

        if rows:

            df = pd.DataFrame(rows)

            try:
                display(df.style.hide(axis="index"))
            except:
                display(df)

        else:
            print("Bucket kosong.")

    object_dropdown.options = (
        options if options
        else [("Tidak ada Object", None)]
    )


# ==========================================
# Upload Object
# ==========================================

def upload_file():
    """Upload file ke bucket yang dipilih"""

    if bucket_dropdown.value is None:
        print("Pilih bucket terlebih dahulu.")
        return

    if len(upload_widget.value) == 0:
        print("Pilih file terlebih dahulu.")
        return

    # FileUpload ipywidgets 8.x
    file_info = upload_widget.value[0]

    file_name = file_info["name"]

    # Konversi memoryview menjadi bytes
    file_content = bytes(file_info["content"])

    try:

        s3.put_object(
            Bucket=bucket_dropdown.value,
            Key=file_name,
            Body=file_content
        )

        print(f"✅ File '{file_name}' berhasil diupload.")

        refresh_object_data()

    except Exception as e:
        print(f"Upload gagal : {e}")


# ==========================================
# Download Object
# ==========================================

def download_object():
    """Download object dari bucket"""

    if bucket_dropdown.value is None:
        print("Pilih bucket.")
        return

    if object_dropdown.value is None:
        print("Pilih object.")
        return

    os.makedirs("downloads", exist_ok=True)

    file_path = os.path.join(
        "downloads",
        object_dropdown.value
    )

    try:

        s3.download_file(
            bucket_dropdown.value,
            object_dropdown.value,
            file_path
        )

        print(f"✅ File berhasil didownload ke:\n{file_path}")

    except Exception as e:
        print(e)


# ==========================================
# Delete Object
# ==========================================

def delete_object():
    """Menghapus object"""

    if bucket_dropdown.value is None:
        print("Pilih bucket.")
        return

    if object_dropdown.value is None:
        print("Pilih object.")
        return

    try:

        s3.delete_object(
            Bucket=bucket_dropdown.value,
            Key=object_dropdown.value
        )

        print(f"✅ {object_dropdown.value} berhasil dihapus.")

        refresh_object_data()

    except Exception as e:
        print(e)

## Event Handler

Menghubungkan setiap tombol dengan fungsi yang sesuai.

In [20]:
# ==========================================
# EVENT HANDLER
# ==========================================

# Create Bucket
def klik_bucket_create(b):

    with bucket_output:

        bucket_output.clear_output()

        create_bucket()


# Refresh Bucket
def klik_bucket_refresh(b):

    refresh_bucket_data()


# Delete Bucket
def klik_bucket_delete(b):

    with bucket_output:

        bucket_output.clear_output()

        delete_bucket()


# Upload Object
def klik_upload(b):

    with object_output:

        object_output.clear_output()

        upload_file()


# Download Object
def klik_download(b):

    with object_output:

        object_output.clear_output()

        download_object()


# Delete Object
def klik_delete_object(b):

    with object_output:

        object_output.clear_output()

        delete_object()


# ==========================================
# BIND BUTTON EVENT
# ==========================================

btn_bucket_create.on_click(klik_bucket_create)
btn_bucket_refresh.on_click(klik_bucket_refresh)
btn_bucket_delete.on_click(klik_bucket_delete)

btn_upload.on_click(klik_upload)
btn_download.on_click(klik_download)
btn_delete_object.on_click(klik_delete_object)

In [21]:
# ==========================================
# LAYOUT S3 BUCKET MANAGER
# ==========================================

# Output Area
bucket_output = widgets.Output()
bucket_table = widgets.Output()
object_table = widgets.Output()
object_output = widgets.Output()

# Panel S3
bucket_panel = widgets.VBox([

    # Judul
    widgets.HTML("<h2>🪣 S3 Bucket Manager</h2>"),

    # Input Nama Bucket
    widgets.HTML("<h4>➕ Membuat Bucket</h4>"),
    bucket_name,

    widgets.HBox([
        btn_bucket_create,
        btn_bucket_refresh
    ]),

    bucket_output,

    widgets.HTML("<hr>"),

    # Daftar Bucket
    widgets.HTML("<h4>📂 Daftar Bucket</h4>"),
    bucket_dropdown,
    bucket_table,

    btn_bucket_delete,

    widgets.HTML("<hr>"),

    # Object Storage
    widgets.HTML("<h4>📄 Object Storage</h4>"),
    object_dropdown,
    object_table,

    widgets.HTML("<br>"),

    # Upload File
    widgets.HTML("<b>Upload File</b>"),
    upload_widget,

    widgets.HBox([
        btn_upload,
        btn_download,
        btn_delete_object
    ]),

    object_output

])

display(bucket_panel)

# Menampilkan data bucket pertama kali
refresh_bucket_data()